In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from itables import init_notebook_mode, show

client = bigquery.Client()

c:\Users\jy\anaconda3\envs\tzesm\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [7]:
query = """
    SELECT *
    FROM `tz-data-dev.taiwan_lnd.power_breakdown`
"""

# Run the query
query_job = client.query(query)

# Convert the result to a pandas DataFrame
df = query_job.to_dataframe()

c:\Users\jy\anaconda3\envs\tzesm\Lib\site-packages\google\cloud\bigquery\table.py:1933: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [8]:
df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M:%S')
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

In [9]:
df_prod_monthly = df[['powerConsumptionTotal','powerProductionTotal', 'production_nuclear',
                      'production_geothermal', 'production_biomass', 'production_coal',
                      'production_wind', 'production_solar', 'production_hydro',
                      'production_gas', 'production_oil', 'production_unknown',
                      'production_hydro discharge']]

In [10]:
df_prod_monthly_2024 = df_prod_monthly.resample('ME').sum().loc['2024']

In [11]:
df_prod_monthly_2024['month'] = df_prod_monthly_2024.index.month

In [13]:
df_monthly_actual_2024 = pd.read_csv('Generation_2024.csv')

In [14]:
def plot_monthly_comparison_with_tolerance(
    actual_df: pd.DataFrame,
    collected_df: pd.DataFrame,
    month_col: str,
    actual_col: str,
    collected_col: str,
    tolerance: float = 0.05,
    title: str = None
) -> go.Figure:
    """
    Plot a grouped bar chart of actual vs. collected monthly values,
    with ±tolerance error bars on the actuals.

    Parameters
    ----------
    actual_df : DataFrame
        Contains the actual production/measurement.
    collected_df : DataFrame
        Contains the collected/reference production/measurement.
    month_col : str
        Name of the column to join on (e.g. 'month').
    actual_col : str
        Column name of the actual values (e.g. 'Coal').
    collected_col : str
        Column name of the collected values (e.g. 'production_coal').
    tolerance : float, default 0.05
        Fractional tolerance for the error bar (0.05 ⇒ ±5%).
    title : str, optional
        Chart title. If None, a generic title will be used.

    Returns
    -------
    fig : go.Figure
        The Plotly Figure object.
    """
    # Merge on month
    merged = (
        actual_df[[month_col, actual_col]]
        .merge(
            collected_df[[month_col, collected_col]],
            on=month_col,
            how='inner'
        )
        .dropna(subset=[actual_col, collected_col])
    )

    # Compute ± tolerance error
    merged['err'] = (merged[actual_col] * tolerance).abs()

    # Build the figure
    fig = go.Figure([
        # Actual with error bars
        go.Bar(
            x=merged[month_col],
            y=merged[actual_col],
            name=f"{actual_col} (Actual)",
            error_y=dict(
                type='data',
                array=merged['err'],
                arrayminus=merged['err']
            )
        ),
        # Collected without error bars
        go.Bar(
            x=merged[month_col],
            y=merged[collected_col],
            name=f"{collected_col} (Collected)"
        )
    ])
    
    fig.update_layout(
        barmode='group',
        title=title or f"Monthly {actual_col} vs {collected_col}  (±{tolerance*100:.0f}% band)",
        xaxis_title=month_col,
        yaxis_title=f"Value ({actual_col})",
        legend=dict(x=0.02, y=0.98)
    )

    return fig


Comparison between Actual (National Statistics) and Collected (Electricity Map)

In [23]:
# Coal Generation comparison
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024, 
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Coal',
                                       collected_col='production_coal',
                                       tolerance=0.1,
                                       title="Monthly Coal Production Comparison (2024)")

In [22]:
# Natural Gas Generation comparison
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024, 
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Gas',
                                       collected_col='production_gas',
                                       tolerance=0.1,
                                       title="Monthly Natural Gas Production Comparison (2024)")

In [21]:
# Oil Generation comparison

plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Oil',
                                       collected_col='production_oil',
                                       tolerance=0.1,
                                       title="Monthly Oil Production Comparison (2024)")

In [31]:
# Nuclear Generation comparison
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Nuclear',
                                       collected_col='production_nuclear',
                                       tolerance=0.1,
                                       title="Monthly Nuclear Production Comparison (2024)")

In [ ]:
# solar Comparison - Comment: First few month of 2024 are missing data thus might be the reason for the error bars to be larger than the actual value.
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Solar',
                                       collected_col='production_solar',
                                       tolerance=0.1,
                                       title="Monthly Solar Production Comparison (2024)")

In [25]:
# Wind Generation comparison
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Wind',
                                       collected_col='production_wind',
                                       tolerance=0.1,
                                       title="Monthly Wind Production Comparison (2024)")

In [26]:
# Hydro Generation comparison

plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Hydro',
                                       collected_col='production_hydro',
                                       tolerance=0.1,
                                       title="Monthly Hydro Production Comparison (2024)")

In [28]:
# Geothermal Generation comparison
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Geothermal',
                                       collected_col='production_geothermal',
                                       tolerance=0.1,
                                       title="Monthly Geothermal Production Comparison (2024)")

In [ ]:
# Biomass Generation comparison - Comment: Biomass reported on national statistics includes contribution from autoproducer which only a varying share of them are being reported
# on Taipower's website. Thus the collected value is lower than the actual value. 
plot_monthly_comparison_with_tolerance(df_monthly_actual_2024,
                                       df_prod_monthly_2024,
                                       month_col='month',
                                       actual_col='Biomass',
                                       collected_col='production_biomass',
                                       tolerance=0.1,
                                       title="Monthly Biomass Production Comparison (2024)")